# Multiple Linear Regression From Scratch Lab
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CreekCS/ai-ml-textbook-labs/blob/main/multiple-linear-regression-from-scratch.ipynb)


# LAB INSTRUCTIONS
To save your work, you **must** create your own copy:
1. Click **File**
2. Select **Save a copy in Drive**
3. Rename the file to `[YourName]_MultipleLinearRegressionLab.ipynb`

In this lab you will build multiple linear regression using gradient descent from scratch.
Rules:
- You may use `pandas` to read the CSV.
- Implement all model functions yourself (prediction, loss, gradients, training loop, scaling).
- You may use scikit-learn only for comparison cells.


## Learning Goals
By the end of this lab you will:
1. Build multiple linear regression with gradient descent from scratch.
2. Train and evaluate the model **without** feature scaling.
3. Compare your result to scikit-learn on the same train/test split.
4. Implement z-score feature scaling yourself.
5. Train again with scaled features and compare performance.


In [ ]:
!pip3 -q install pandas scikit-learn

In [ ]:
import pandas as pd
from sklearn.linear_model import LinearRegression

DATA_URL = "https://raw.githubusercontent.com/CreekCS/ai-ml-textbook-labs/main/house_prices_linear.csv"


## Step 1: Load and Inspect Data
Only `pandas` is allowed for CSV loading.


In [ ]:
df = pd.read_csv(DATA_URL)
df.head()


In [ ]:
print("Rows:", len(df))
print("Columns:", list(df.columns))


## Step 2: Build Train/Test Split (No Helpers)
Use the first 80% of rows as train and the last 20% as test.


In [ ]:
FEATURE_COLS = ["sqft", "bedrooms", "age"]
TARGET_COL = "price"

X_all = df[FEATURE_COLS].values.tolist()
y_all = df[TARGET_COL].tolist()

split_idx = int(0.8 * len(df))
X_train = X_all[:split_idx]
y_train = y_all[:split_idx]
X_test = X_all[split_idx:]
y_test = y_all[split_idx:]

print("Train size:", len(X_train))
print("Test size:", len(X_test))


## Step 3: Implement Core Functions
Complete each TODO. Do not use NumPy linear algebra helpers.


In [ ]:
def predict_row(x_row, weights, bias):
    # TODO: return dot(weights, x_row) + bias
    pass


def predict_batch(X, weights, bias):
    # TODO: use predict_row for each row and return list of predictions
    pass


def compute_mse(y_true, y_pred):
    # TODO: return mean squared error
    pass


In [ ]:
def compute_gradients(X, y, weights, bias):
    # Returns grad_w (list) and grad_b (float)
    # TODO:
    # 1) Loop over all examples
    # 2) error = prediction - y[i]
    # 3) grad_w[j] += (2/n) * error * X[i][j]
    # 4) grad_b += (2/n) * error
    pass


In [ ]:
def train_linear_regression(X, y, learning_rate=1e-8, epochs=5000):
    # TODO:
    # 1) initialize weights to zeros, bias to 0.0
    # 2) repeat gradient descent updates for epochs
    # 3) every 1000 epochs compute and store training loss in history
    # 4) return weights, bias, history
    pass


## Step 4: Train From Scratch Without Feature Scaling
Try the provided defaults first. If loss explodes, lower `learning_rate`.


In [ ]:
weights_raw, bias_raw, history_raw = train_linear_regression(
    X_train,
    y_train,
    learning_rate=1e-8,
    epochs=20000,
)

print("Weights (raw):", weights_raw)
print("Bias (raw):", bias_raw)
print("Loss checkpoints (epoch, mse):")
for item in history_raw[:5]:
    print(item)
print("...")
print(history_raw[-1])


In [ ]:
y_pred_test_raw = predict_batch(X_test, weights_raw, bias_raw)
mse_test_raw = compute_mse(y_test, y_pred_test_raw)
print("Scratch test MSE (no scaling):", mse_test_raw)


## Step 5: Compare Weights Against scikit-learn (No Scaling)
Use the same train/test split, then compare learned weights and bias.


In [ ]:
sk_model_raw = LinearRegression()
sk_model_raw.fit(X_train, y_train)

sk_weights_raw = sk_model_raw.coef_.tolist()
sk_bias_raw = float(sk_model_raw.intercept_)

print("Scratch weights (no scaling):", weights_raw)
print("sklearn weights (no scaling):", sk_weights_raw)
print("Scratch bias (no scaling):", bias_raw)
print("sklearn bias (no scaling):", sk_bias_raw)


In [ ]:
feature_weight_diffs_raw = []
for j in range(len(weights_raw)):
    feature_weight_diffs_raw.append(abs(weights_raw[j] - sk_weights_raw[j]))

bias_diff_raw = abs(bias_raw - sk_bias_raw)

print("Absolute weight differences by feature (no scaling):")
for name, diff in zip(FEATURE_COLS, feature_weight_diffs_raw):
    print(f"  {name}: {diff}")
print("Absolute bias difference (no scaling):", bias_diff_raw)


## Step 6: Implement Z-Score Feature Scaling
Scale each feature column using training-set stats only.

Formula per feature:
\[ z = \frac{x - \mu}{\sigma} \]

If a feature has `std = 0`, use `1.0` instead to avoid divide-by-zero.


In [ ]:
def compute_feature_stats(X):
    # TODO: return (means, stds)
    # means[j] = average of column j
    # stds[j] = population std dev of column j
    pass


def zscore_transform(X, means, stds):
    # TODO: return transformed X as list of lists
    pass


In [ ]:
means, stds = compute_feature_stats(X_train)

X_train_scaled = zscore_transform(X_train, means, stds)
X_test_scaled = zscore_transform(X_test, means, stds)

print("Feature means:", means)
print("Feature stds:", stds)


## Step 7: Train From Scratch With Scaled Features
Because scaling changes gradient magnitude, use a larger learning rate.


In [ ]:
weights_scaled, bias_scaled, history_scaled = train_linear_regression(
    X_train_scaled,
    y_train,
    learning_rate=0.01,
    epochs=20000,
)

y_pred_test_scaled = predict_batch(X_test_scaled, weights_scaled, bias_scaled)
mse_test_scaled = compute_mse(y_test, y_pred_test_scaled)

print("Scratch test MSE (with z-score scaling):", mse_test_scaled)


## Step 8: Compare Scaled Weights vs Scaled scikit-learn
Compare weights and bias after z-score scaling.


In [ ]:
sk_model_scaled = LinearRegression()
sk_model_scaled.fit(X_train_scaled, y_train)

sk_weights_scaled = sk_model_scaled.coef_.tolist()
sk_bias_scaled = float(sk_model_scaled.intercept_)

print("Scratch weights (scaled features):", weights_scaled)
print("sklearn weights (scaled features):", sk_weights_scaled)
print("Scratch bias (scaled features):", bias_scaled)
print("sklearn bias (scaled features):", sk_bias_scaled)

feature_weight_diffs_scaled = []
for j in range(len(weights_scaled)):
    feature_weight_diffs_scaled.append(abs(weights_scaled[j] - sk_weights_scaled[j]))

bias_diff_scaled = abs(bias_scaled - sk_bias_scaled)

print("Absolute weight differences by feature (scaled):")
for name, diff in zip(FEATURE_COLS, feature_weight_diffs_scaled):
    print(f"  {name}: {diff}")
print("Absolute bias difference (scaled):", bias_diff_scaled)


## Step 9: Reflection Questions
Write short answers in a new markdown cell:
1. How close were your scratch weights and sklearn weights before scaling?
2. After z-score scaling, did the weight differences become smaller or larger? Why?
3. Why do we compute scaling stats on train only (not full dataset)?
4. Which feature had the largest absolute weight in each experiment, and what does that suggest?
